# Language-Model Fine-Tuning Tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_finetuning_colab.ipynb)

This notebook is an educational, end-to-end tutorial on supervised fine-tuning (SFT) for causal language models.

> approved pinned base → Hugging Face or local ZIP snapshot → sample or custom dataset → validation → baseline generation → QLoRA SFT → before/after generation → novel prompt evaluation → adapter export → fresh reload

## What you will learn

In this tutorial, you will explore how a **base model**, **tokenizer**, and **PEFT adapter** operate together as a cohesive system:
- Why immutable revisions and cryptographic pinning matter for reproducible machine learning;
- How conversational chat templates transform multi-turn dialogs into structured input token sequences;
- Why we apply **assistant-only** loss masking so optimization focuses exclusively on model answers rather than user questions;
- How 4-bit NormalFloat quantization (QLoRA) makes fine-tuning multi-billion parameter models feasible on standard consumer GPUs;
- How to interpret **optimization** metrics (loss, perplexity, gradient norms) without conflating them with subjective **task quality**; and
- Why verifying a fresh model reload from exported disk artifacts is essential for ensuring deployment readiness.

Throughout this guide, we maintain a clear separation between **pipeline execution** (verifying that training ran without errors), **optimization** evidence (measured loss convergence, perplexity, and memory usage), and actual **task quality** (whether the model truly fulfills your intended use case). The prompt comparisons in this tutorial illustrate behavioral shifts during fine-tuning.

**Default candidate:** Qwen3 0.6B. **AI provenance:** OpenAI / ChatGPT — GPT-5.6 Sol High, Builder role.


## 1. Hardware, GPU Runtime, and Dependencies

Fine-tuning a causal language model is computationally intensive and memory-bound. Supervised fine-tuning requires tracking activations, gradients, optimizer states, and temporary attention matrices. Standard 16-bit fine-tuning of even a 3B model can easily exceed 24 GB of GPU memory.

To make fine-tuning accessible on standard single-GPU environments (such as Google Colab's T4, V100, or A100 tiers), this tutorial employs **QLoRA** (Quantized Low-Rank Adaptation). Under QLoRA:
- The base model weights are loaded and frozen in 4-bit NormalFloat (`nf4`) format;
- Trainable low-rank adapter matrices are attached to attention projection layers; and
- Memory consumption during backward passes is reduced by over 60%, allowing models up to 4B parameters to comfortably train within 8 GB to 16 GB of VRAM.

The library versions below (`transformers`, `tokenizers`, `peft`, `bitsandbytes`, `accelerate`) are pinned to guarantee consistent chat template formatting, reliable 4-bit kernel dispatch, and reproducible execution across environments.


In [ ]:
%pip -q install transformers==4.57.1 tokenizers==0.22.1 huggingface-hub==0.36.0 peft==0.18.0 accelerate==1.11.0 bitsandbytes==0.49.0 safetensors==0.8.0 "datasets>=3,<5"


In [ ]:
import gc,hashlib,json,math,os,platform,random,re,shutil,stat,time,zipfile
from pathlib import Path,PurePosixPath
import pandas as pd,torch
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from peft import LoraConfig,PeftModel,get_peft_model,prepare_model_for_kbit_training
if not torch.cuda.is_available(): raise RuntimeError("Use a Colab GPU runtime")
GPU_NAME=torch.cuda.get_device_name(0); GPU_VRAM_GB=torch.cuda.get_device_properties(0).total_memory/1024**3
AI_PROVENANCE={"provider":"OpenAI","product":"ChatGPT","model":"GPT-5.6 Sol High","role":"Builder","note":"Not independent reviewer sign-off"}
print(platform.python_version(),torch.__version__,GPU_NAME,f"{GPU_VRAM_GB:.2f} GiB")


## 2. Choose a Base Model

In Supervised Fine-Tuning, the base model provides the underlying language understanding, reasoning capabilities, and world knowledge. The fine-tuning process then adapts its tone, formatting, and instruction adherence.

This tutorial restricts selection to approved, verified causal language models pinned to immutable 40-character commit SHAs. Fixing the exact commit hash ensures supply-chain integrity, prevents silent upstream changes, and guarantees that `trust_remote_code=False` is maintained for safety.

You can choose from several distinct open-weights model families:
- **`qwen3-0.6b`** — **Default tutorial candidate**; compact 0.6B smoke model (~1.41 GiB) for rapid verification on free-tier Colab GPUs (Apache-2.0).
- **`smollm3-3b`** — Balanced 3B architecture from Hugging Face (Apache-2.0).
- **`qwen3-1.7b`** & **`qwen3-4b`** — Powerful Qwen 3 language models with strong reasoning and multi-turn conversational performance (Apache-2.0).
- **`granite-4.1-3b`** & **`granite-3.1-2b-instruct`** — IBM Granite models built with full enterprise transparency, audited data governance, and strong tool-calling support (Apache-2.0).
- **`deepseek-r1-distill-qwen-1.5b`** — Compact reasoning model fine-tuned on reasoning traces; outputs explicit `<think>...</think>` thoughts under a permissive MIT license.
- **`qwen2.5-coder-1.5b`** — Highly capable code, SQL, and structured JSON specialist under Apache-2.0.
- **`smollm2-1.7b`** & **`smollm2-360m`** — Lightweight models trained on curated educational datasets (Cosmopedia v2, FineWeb-Edu); fast to train and ideal for resource-constrained environments.
- **`h2o-danube3-4b-chat`** — Efficient mobile and edge-optimized architecture from H2O.ai (Apache-2.0).
- **`llama-3.2-3b-instruct`** — Gated candidate requiring accepted terms and a Hugging Face token.

For **Llama 3.2 3B Instruct**, the repository is gated on Hugging Face. When selecting this model, provide your `HF_TOKEN` in Colab Secrets and enable Notebook access. A successful run demonstrates authenticating and fine-tuning a gated model checkpoint using your credentials in Colab.


In [ ]:
TUTORIAL_REGISTRY={
"qwen3-0.6b":{"model_id":"Qwen/Qwen3-0.6B","revision":"c1899de289a04d12100db370d81485cdf75e47ca","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"smoke-ci/apache-2.0"},
"smollm3-3b":{"model_id":"HuggingFaceTB/SmolLM3-3B","revision":"a07cc9a04f16550a088caea529712d1d335b0ac1","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"tutorial-candidate/internal-only"},
"qwen3-1.7b":{"model_id":"Qwen/Qwen3-1.7B","revision":"70d244cc86ccca08cf5af4e1e306ecf908b1ad5e","license":"apache-2.0","min_vram_gb":9.3,"requires_hf_token":False,"dimer_zip":True,"state":"user-facing"},
"qwen3-4b":{"model_id":"Qwen/Qwen3-4B","revision":"1cfa9a7208912126459214e8b04321603b3df60c","license":"apache-2.0","min_vram_gb":11.5,"requires_hf_token":False,"dimer_zip":True,"state":"user-facing"},
"granite-4.1-3b":{"model_id":"ibm-granite/granite-4.1-3b","revision":"c0650403e44e78ec0262dab1c90914c65b196c4e","license":"apache-2.0","min_vram_gb":8.7,"requires_hf_token":False,"dimer_zip":True,"state":"user-facing"},
"deepseek-r1-distill-qwen-1.5b":{"model_id":"deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B","revision":"ad9f0ae0864d7fbcd1cd905e3c6c5b069cc8b562","license":"mit","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"open-reasoning/mit"},
"qwen2.5-coder-1.5b":{"model_id":"Qwen/Qwen2.5-Coder-1.5B-Instruct","revision":"2e1fd397ee46e1388853d2af2c993145b0f1098a","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"open-code/apache-2.0"},
"smollm2-1.7b":{"model_id":"HuggingFaceTB/SmolLM2-1.7B-Instruct","revision":"31b70e2e869a7173562077fd711b654946d38674","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"open-small/apache-2.0"},
"smollm2-360m":{"model_id":"HuggingFaceTB/SmolLM2-360M-Instruct","revision":"a10cc1512eabd3dde888204e902eca88bddb4951","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"open-small/apache-2.0"},
"granite-3.1-2b-instruct":{"model_id":"ibm-granite/granite-3.1-2b-instruct","revision":"bbc2aed595bd38bd770263dc3ab831db9794441d","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"enterprise-transparent/apache-2.0"},
"h2o-danube3-4b-chat":{"model_id":"h2oai/h2o-danube3-4b-chat","revision":"1e5c6fa6620f8bf078958069ab4581cd88e0202c","license":"apache-2.0","min_vram_gb":None,"requires_hf_token":False,"dimer_zip":True,"state":"mobile-edge/apache-2.0"},
"llama-3.2-3b-instruct":{"model_id":"meta-llama/Llama-3.2-3B-Instruct","revision":"0cb88a4f764b7a12671c53f0838cd831a0843b95","license":"llama3.2","min_vram_gb":None,"requires_hf_token":True,"dimer_zip":False,"state":"credential-test-candidate"}}
BASE_MODEL_KEY = "qwen3-0.6b" # @param ["qwen3-0.6b","smollm3-3b","qwen3-1.7b","qwen3-4b","granite-4.1-3b","deepseek-r1-distill-qwen-1.5b","qwen2.5-coder-1.5b","smollm2-1.7b","smollm2-360m","granite-3.1-2b-instruct","h2o-danube3-4b-chat","llama-3.2-3b-instruct"]
MODEL_SOURCE = "Pinned Hugging Face" # @param ["Pinned Hugging Face","DIMER ZIP"]
TRAINING_METHOD = "qlora" # @param ["qlora"]
MAX_SEQUENCE_LENGTH = 512 # @param {type:"integer"}
EPOCHS = 1 # @param {type:"integer"}
LEARNING_RATE = 0.0002 # @param {type:"number"}
LORA_RANK = 8 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}
entry=TUTORIAL_REGISTRY[BASE_MODEL_KEY]; model_id=entry["model_id"]; revision=entry["revision"]; base_license=entry["license"]
if MODEL_SOURCE=="DIMER ZIP" and not entry["dimer_zip"]: raise RuntimeError("This model is not approved for the DIMER ZIP path; use Pinned Hugging Face.")
if entry["min_vram_gb"] is None: print("⚠ Unmeasured tutorial candidate: this run records observed VRAM; it does not set a production minimum.")
elif GPU_VRAM_GB<entry["min_vram_gb"]: raise RuntimeError(f"Need at least {entry['min_vram_gb']} GiB for the measured profile")
print(BASE_MODEL_KEY,model_id,revision,entry["state"])


## 3. Model Acquisition, Credentials, and Snapshot Verification

Public open-weights checkpoints require no credentials to download. For gated models (such as Llama 3.2), the notebook reads an `HF_TOKEN` from Colab Secrets and performs an authenticated metadata preflight check before downloading weights. The token is used solely in memory and is never printed or saved into output artifacts.

Two model acquisition modes are supported:
- **`Pinned Hugging Face`** — Downloads the exact immutable commit revision directly from the Hugging Face Hub using `snapshot_download`.
- **`DIMER ZIP`** — Loads an offline base-model package (such as one created using our local CLI fetch tool). To protect against corrupt or tampered files, the snapshot package must contain a `dimer-base-manifest.json` file with format `dimer_hf_snapshot`. The extraction process verifies path safety, checks that every listed file matches its recorded byte count and SHA-256 hash, and confirms the presence of standard `.safetensors` weights.

Enforcing `trust_remote_code=False` ensures that model loading never executes arbitrary Python code from untrusted external repositories.


In [ ]:
HF_TOKEN=None
if entry["requires_hf_token"]:
 from google.colab import userdata
 try: HF_TOKEN=userdata.get("HF_TOKEN")
 except Exception as exc: raise RuntimeError("Add HF_TOKEN in Colab Secrets, enable Notebook access, and accept the model terms.") from exc
 info=HfApi(token=HF_TOKEN).model_info(model_id,revision=revision)
 if info.sha!=revision: raise RuntimeError("Pinned Hugging Face revision mismatch")
 print("✓ Hugging Face credential/revision preflight passed.")

UPLOAD_DIMER_ZIP = False # @param {type:"boolean"}
DIMER_ZIP_PATH = "/content/dimer-base-model.zip" # @param {type:"string"}
EXPECTED_DIMER_ZIP_SHA256 = "" # @param {type:"string"}

def sha256_file(p):
 h=hashlib.sha256()
 with open(p,"rb") as f:
  for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
 return h.hexdigest()
def member(root,name):
 if "\\" in name: raise ValueError("Unsafe ZIP path")
 q=PurePosixPath(name)
 if q.is_absolute() or ".." in q.parts: raise ValueError("Unsafe ZIP path")
 p=(Path(root).resolve()/Path(*q.parts)).resolve()
 if p!=Path(root).resolve() and Path(root).resolve() not in p.parents: raise ValueError("ZIP escape")
 return p
def verify_dimer_zip(path):
 path=Path(path); outer=sha256_file(path)
 if EXPECTED_DIMER_ZIP_SHA256 and outer.lower()!=EXPECTED_DIMER_ZIP_SHA256.strip().lower(): raise ValueError("DIMER ZIP SHA-256 mismatch")
 root=Path("/content/dimer-base-model").resolve(); shutil.rmtree(root,ignore_errors=True); root.mkdir()
 total=0
 with zipfile.ZipFile(path) as z:
  for i in z.infolist():
   if stat.S_ISLNK((i.external_attr>>16)&0xffff): raise ValueError("Symlink not allowed")
   total+=i.file_size
   if total>20*1024**3: raise ValueError("DIMER ZIP expands beyond 20 GiB")
   p=member(root,i.filename)
   if i.is_dir(): p.mkdir(parents=True,exist_ok=True); continue
   p.parent.mkdir(parents=True,exist_ok=True)
   with z.open(i) as src,open(p,"wb") as dst: shutil.copyfileobj(src,dst)
 manifests=list(root.rglob("dimer-base-manifest.json"))
 if len(manifests)!=1: raise ValueError("Expected one dimer-base-manifest.json")
 model_root=manifests[0].parent; m=json.loads(manifests[0].read_text())
 if m.get("format")!="dimer_hf_snapshot" or m.get("formatVersion")!=1: raise ValueError("Unsupported DIMER snapshot format")
 if (m.get("modelKey"),m.get("modelId"),m.get("revision"))!=(BASE_MODEL_KEY,model_id,revision): raise ValueError("DIMER model identity mismatch")
 listed=set(); size=0
 for r in m.get("files",[]):
  p=member(model_root,r["path"])
  if not p.is_file() or p.stat().st_size!=r["bytes"] or sha256_file(p)!=r["sha256"]: raise ValueError(f"DIMER file verification failed: {r['path']}")
  listed.add(r["path"]); size+=r["bytes"]
 actual={p.relative_to(model_root).as_posix() for p in model_root.rglob("*") if p.is_file() and p.name!="dimer-base-manifest.json"}
 if actual!=listed or size!=m.get("totalBytes"): raise ValueError("DIMER manifest/file-set mismatch")
 if not any(x.endswith(".safetensors") for x in listed): raise ValueError("No safetensors weights")
 return model_root,outer

MODEL_LOAD_REF=model_id; BASE_MODEL_ACQUISITION={"source":MODEL_SOURCE,"modelId":model_id,"revision":revision}
if MODEL_SOURCE=="DIMER ZIP":
 if UPLOAD_DIMER_ZIP:
  from google.colab import files
  u=files.upload()
  if len(u)!=1: raise ValueError("Upload exactly one DIMER ZIP")
  Path(DIMER_ZIP_PATH).write_bytes(next(iter(u.values())))
 MODEL_LOAD_REF,dsha=verify_dimer_zip(DIMER_ZIP_PATH); BASE_MODEL_ACQUISITION.update({"packageFormat":"dimer_hf_snapshot","zipSha256":dsha})
 print("✓ DIMER base-model package verified")


## 4. Dataset Formatting and Structural Validation

Supervised Fine-Tuning trains a model to produce desired assistant responses when presented with specific conversational context. Unlike tabular machine learning where rows are fixed columns, SFT datasets consist of conversational turns.

In this tutorial, dataset examples are normalized into a canonical `messages` structure containing `role` (`system`, `user`, or `assistant`) and `content` fields. The normalization handles three common schemas:
1. **Chat records**: Lists of conversational messages.
2. **Prompt and Completion**: Simple single-turn question-answer pairs.
3. **Instruction, Input, and Output**: Alpaca-style instruction records with optional context.

Data hygiene is enforced before any training begins:
- **Assistant Targets**: Every training record must contain at least one non-empty assistant response to supervise.
- **Split Separation**: Training, validation, and optional test splits are isolated.
- **Leakage Detection**: The notebook computes SHA-256 digests of canonical records and halts immediately if identical examples appear in both training and validation splits.
- **Transparent Handling**: Exact duplicates within a split are reported for user awareness rather than quietly erased.


In [ ]:
DATA_SOURCE = "Sample: Filipino SFT" # @param ["Sample: Filipino SFT","Sample: Dolly","Bring Your Own Dataset"]
SAMPLE_LIMIT = 120 # @param {type:"integer"}
MAX_TOTAL_TRAIN_TOKENS = 50_000_000
W=Path("/content/lm-sft"); shutil.rmtree(W,ignore_errors=True); W.mkdir()
def canonical(r):
 if "messages" in r: m=[{"role":x["role"],"content":str(x["content"])} for x in r["messages"]]
 elif "prompt" in r and ("completion" in r or "response" in r): m=[{"role":"user","content":str(r["prompt"])},{"role":"assistant","content":str(r.get("completion",r.get("response")))}]
 elif "instruction" in r and ("output" in r or "response" in r):
  q=str(r["instruction"])+(f"\n\n{r.get('input') or r.get('context')}" if r.get("input") or r.get("context") else ""); m=[{"role":"user","content":q},{"role":"assistant","content":str(r.get("output",r.get("response")))}]
 else: raise ValueError("Unsupported SFT schema")
 if any(x["role"] not in {"system","user","assistant"} for x in m) or not any(x["role"]=="assistant" and x["content"].strip() for x in m): raise ValueError("Invalid roles/assistant target")
 return {"messages":m}
def fp(r): return hashlib.sha256(json.dumps(r,sort_keys=True,ensure_ascii=False,separators=(",",":")).encode()).hexdigest()
if DATA_SOURCE.startswith("Sample"):
 dsid="jpaulpoliquit/ph-sft-ai-authored-v1" if DATA_SOURCE=="Sample: Filipino SFT" else "databricks/databricks-dolly-15k"
 rev=HfApi().dataset_info(dsid).sha if "ph-sft" in dsid else "bdd27f4d94b9c1f951818a7da7fd7aeea5dbff1a"
 lic="apache-2.0" if "ph-sft" in dsid else "cc-by-sa-3.0"
 rows=sorted([canonical(dict(x)) for x in load_dataset(dsid,revision=rev,split="train")],key=fp)[:SAMPLE_LIMIT]; cut=max(1,len(rows)//5)
 SPLITS={"train":rows[cut:],"validation":rows[:cut]}; DATASET_PROVENANCE={"source":dsid,"revision":rev,"license":lic,"usage":"tutorial-training-not-benchmark"}
else:
 from google.colab import files
 u=files.upload(); root=W/"byod"; root.mkdir()
 if len(u)==1 and next(iter(u)).lower().endswith(".zip"):
  zp=W/"data.zip"; zp.write_bytes(next(iter(u.values())))
  with zipfile.ZipFile(zp) as z:
   for i in z.infolist():
    p=member(root,i.filename)
    if stat.S_ISLNK((i.external_attr>>16)&0xffff): raise ValueError("Unsafe dataset ZIP")
    if i.is_dir(): p.mkdir(parents=True,exist_ok=True); continue
    p.parent.mkdir(parents=True,exist_ok=True)
    with z.open(i) as src,open(p,"wb") as dst: shutil.copyfileobj(src,dst)
 else:
  for n,b in u.items(): (root/Path(n).name).write_bytes(b)
 def read(p): return [canonical(json.loads(x)) for x in p.read_text().splitlines() if x.strip()]
 if not (root/"train.jsonl").exists(): raise ValueError("BYOD requires train.jsonl")
 if (root/"validation.jsonl").exists() and (root/"val.jsonl").exists(): raise ValueError("Ambiguous validation split")
 SPLITS={"train":read(root/"train.jsonl")}; v=root/("validation.jsonl" if (root/"validation.jsonl").exists() else "val.jsonl")
 if v.exists(): SPLITS["validation"]=read(v)
 if (root/"test.jsonl").exists(): SPLITS["test"]=read(root/"test.jsonl")
 DATASET_PROVENANCE={"source":"BYOD","usage":"user-provided"}
for k,v in SPLITS.items():
 if len(v)!=len({fp(x) for x in v}): print(f"Warning: exact duplicates in {k}; none removed")
for a,b in [("train","validation"),("train","test"),("validation","test")]:
 if a in SPLITS and b in SPLITS and {fp(x) for x in SPLITS[a]}&{fp(x) for x in SPLITS[b]}: raise ValueError(f"Split leakage {a}/{b}")
if "validation" not in SPLITS: rows=SPLITS["train"]; cut=max(1,len(rows)//5); SPLITS={**SPLITS,"train":rows[cut:],"validation":rows[:cut]}
DATASET_DIGEST=hashlib.sha256("".join(fp(x) for k in sorted(SPLITS) for x in SPLITS[k]).encode()).hexdigest(); print({k:len(v) for k,v in SPLITS.items()},DATASET_DIGEST[:16])


## 5. Tokenizer Mechanics and Assistant-Only Loss Masking

A tokenizer translates natural language into integer token sequences using a vocabulary and merges table. It also formats conversational roles using the model's specialized chat template (e.g., ChatML or Llama formatting tags).

A fundamental principle of effective Supervised Fine-Tuning is **assistant-only loss masking**:
- If we computed loss across the entire sequence, the optimizer would expend capacity teaching the model to predict the user's prompt.
- To prevent this, we assign non-assistant tokens a label value of `-100`, which is PyTorch's default `ignore_index` for cross-entropy loss.
- Only the tokens generated by the assistant receive active target labels, ensuring that gradients only update parameters to improve answer quality.

To compute assistant token spans accurately, we incrementally render conversation prefixes using the tokenizer's chat template and check that tokenization is prefix-stable. Furthermore, any example that exceeds the maximum sequence length is flagged and rejected rather than silently truncated, preserving complete conversational context.


In [ ]:
tok_kw={"trust_remote_code":False}
if MODEL_SOURCE=="Pinned Hugging Face": tok_kw.update({"revision":revision,**({"token":HF_TOKEN} if HF_TOKEN else {})})
else: tok_kw["local_files_only"]=True
tokenizer=AutoTokenizer.from_pretrained(MODEL_LOAD_REF,**tok_kw)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
if not tokenizer.chat_template: raise ValueError("No chat template")
IGNORE=-100
def enc(s): return tokenizer(s,add_special_tokens=False)["input_ids"]
def rend(m,g=False): return tokenizer.apply_chat_template(m,tokenize=False,add_generation_prompt=g)
def build_masked_example(r):
 m=r["messages"]; ids=enc(rend(m)); labels=[IGNORE]*len(ids)
 if len(ids)>MAX_SEQUENCE_LENGTH: raise ValueError("DATASET_SEQUENCE_TOO_LONG")
 for i,x in enumerate(m):
  if x["role"]=="assistant":
   a,b=enc(rend(m[:i],True)),enc(rend(m[:i+1]))
   if ids[:len(a)]!=a or ids[:len(b)]!=b: raise ValueError("Chat template not prefix-stable")
   labels[len(a):len(b)]=ids[len(a):len(b)]
 if all(x==IGNORE for x in labels): raise ValueError("No supervised tokens")
 return ids,labels
MASKED={k:[build_masked_example(r) for r in v] for k,v in SPLITS.items()}; totals={k:sum(len(x[0]) for x in v) for k,v in MASKED.items()}
if totals["train"]>MAX_TOTAL_TRAIN_TOKENS: raise ValueError("DATASET_TOKEN_BUDGET_EXCEEDED")
print(totals)


## 6. Baseline Generation, QLoRA SFT, and Optimization Evidence

Before initiating training, we record the frozen base model's responses to fixed prompt probes. Capturing this baseline provides a ground truth against which we can observe subsequent behavioral changes.

We then initialize Low-Rank Adaptation (LoRA) on the quantized base model:
- The base model weights remain frozen in 4-bit precision;
- Small, trainable rank decomposition matrices (rank $r=8$, alpha $lpha=16$) are injected into key linear attention layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`, etc.);
- Gradients and AdamW optimizer states are tracked solely for the low-rank adapter parameters, dramatically reducing VRAM overhead.

During fine-tuning, the loop tracks training loss, validation loss, validation perplexity ($e^{	ext{loss}}$), wall-clock duration, and peak allocated GPU memory.

> **Optimization vs. Task Quality:** These metrics measure optimization convergence—namely, how effectively the model minimizes cross-entropy loss on the supervised token distribution. A lower loss indicates successful training, but it does not by itself prove general intelligence, factuality, or domain suitability. Evaluating true **task quality** requires held-out task benchmarks or human evaluation.


In [ ]:
bf16=torch.cuda.is_bf16_supported(); dtype=torch.bfloat16 if bf16 else torch.float16
q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
load_kw={"trust_remote_code":False,"dtype":dtype,"quantization_config":q,"device_map":{"":0}}
if MODEL_SOURCE=="Pinned Hugging Face": load_kw.update({"revision":revision,**({"token":HF_TOKEN} if HF_TOKEN else {})})
else: load_kw["local_files_only"]=True
base_model=AutoModelForCausalLM.from_pretrained(MODEL_LOAD_REF,**load_kw)
def generate(m,p,n=96):
 s=rend([{"role":"user","content":p}],True); x=tokenizer(s,return_tensors="pt",add_special_tokens=False).to("cuda")
 with torch.no_grad(): y=m.generate(**x,max_new_tokens=n,do_sample=False,pad_token_id=tokenizer.pad_token_id)
 return tokenizer.decode(y[0,x["input_ids"].shape[1]:],skip_special_tokens=True).strip()
PROMPTS=["Ipaliwanag sa simpleng Filipino kung ano ang machine learning.","Magbigay ng tatlong paraan para mabawasan ang basura sa opisina."]
BASELINE_OUTPUTS=[generate(base_model,p) for p in PROMPTS]
targets=sorted({n.rsplit(".",1)[-1] for n,_ in base_model.named_modules()}&{"q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"})
if not targets: raise RuntimeError("No LoRA target modules found")
model=get_peft_model(prepare_model_for_kbit_training(base_model),LoraConfig(r=LORA_RANK,lora_alpha=16,lora_dropout=.05,bias="none",task_type="CAUSAL_LM",target_modules=targets))
def batch(x):
 ids,lab=x; return {"input_ids":torch.tensor([ids],device="cuda"),"labels":torch.tensor([lab],device="cuda"),"attention_mask":torch.ones((1,len(ids)),dtype=torch.long,device="cuda")}
def evaluate(xs):
 if not xs: return None
 model.eval(); loss=toks=0
 with torch.no_grad():
  for x in xs:
   b=batch(x); o=model(**b); n=int((b["labels"]!=IGNORE).sum()); loss+=float(o.loss)*n; toks+=n
 model.train(); return loss/toks if toks else None
opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LEARNING_RATE); started=time.time(); torch.cuda.reset_peak_memory_stats(); random.seed(SEED)
for e in range(EPOCHS):
 xs=MASKED["train"][:]; random.shuffle(xs); loss=toks=0; opt.zero_grad()
 for i,x in enumerate(xs,1):
  b=batch(x); o=model(**b); (o.loss/2).backward(); n=int((b["labels"]!=IGNORE).sum()); loss+=float(o.loss)*n; toks+=n
  if i%2==0 or i==len(xs): opt.step(); opt.zero_grad()
 val=evaluate(MASKED["validation"]); print(e+1,loss/toks,val)
METRICS={"trainLoss":loss/toks,"validationLoss":val,"testLoss":evaluate(MASKED.get("test",[])),"validationPerplexity":math.exp(val) if val is not None and val<20 else None,"wallSeconds":time.time()-started,"peakGpuMemoryBytes":torch.cuda.max_memory_allocated(),"peakGpuMemoryGiB":torch.cuda.max_memory_allocated()/1024**3}
ADAPTED_OUTPUTS=[generate(model,p) for p in PROMPTS]
display(pd.DataFrame({"prompt":PROMPTS,"base":BASELINE_OUTPUTS,"adapted":ADAPTED_OUTPUTS})); print(METRICS)


## 7. Interactive Exploration on Novel Prompts

After fine-tuning completes, evaluating the adapted model on the initial baseline prompts reveals how its responses have changed compared to the pre-training baseline.

However, real-world deployment requires testing how the model responds to novel, unseen instructions. This section allows you to interactively test arbitrary prompts to inspect:
- Adherence to instructed formatting and tone;
- Language consistency and response structure (such as step-by-step reasoning or concise summarization); and
- Any signs of repetitive loops or degradation.

While interactive qualitative probing is helpful for rapid feedback during development, formal applications should always complement it with systematic benchmark testing.


In [ ]:
RUN_NEW_PROMPT_INFERENCE = True # @param {type:"boolean"}
NEW_PROMPTS=["Sumulat ng maikling payo para sa isang estudyanteng nagsisimula sa AI.","Ipaliwanag ang pagkakaiba ng training data at evaluation data sa dalawang pangungusap."]
if RUN_NEW_PROMPT_INFERENCE: display(pd.DataFrame({"prompt":NEW_PROMPTS,"response":[generate(model,p) for p in NEW_PROMPTS]}))


## 8. Adapter Export, Provenance Tracking, and Clean Reload

The primary deliverable of QLoRA is the trained **PEFT adapter bundle**, not a duplicate multi-gigabyte copy of the base model weights.

In this step, we export a self-contained, deployment-ready adapter package:
- **Adapter Weights**: Serialized in safe `.safetensors` format;
- **Tokenizer Artifacts**: Vocabularies, configurations, and chat templates;
- **Metrics & Provenance**: Recorded training hyperparameters, runtime environment details, dataset digests, and immutable base-model commit references; and
- **Integrity Manifest**: An `artifact-manifest.json` recording byte counts and SHA-256 hashes for every file in the package.

To verify that the saved artifact is completely standalone and loadable, we execute a **clean reload test**:
1. We explicitly purge the in-memory training model and clear GPU cache;
2. We reacquire the base model from source;
3. We attach the PEFT adapter **from disk**; and
4. We generate a verification response to prove the saved files are sufficient to reconstruct the adapted model.


In [ ]:
S=Path("/content/dimer-lm-adapter.staging"); A=Path("/content/dimer-lm-adapter"); Z=Path("/content/dimer-language-model-adapter.zip")
shutil.rmtree(S,ignore_errors=True); shutil.rmtree(A,ignore_errors=True); S.mkdir()
model.save_pretrained(S,safe_serialization=True); tokenizer.save_pretrained(S/"tokenizer")
PROVENANCE={"artifactFormat":"peft_adapter","artifactFormatVersion":1,"baseModel":model_id,"baseModelRevision":revision,"baseModelRevisionExpected":revision,"baseModelLicense":base_license,"modelKey":BASE_MODEL_KEY,"trustRemoteCode":False,"requiresHfToken":bool(entry["requires_hf_token"]),"dimerZipAllowed":bool(entry["dimer_zip"]),"baseModelAcquisition":BASE_MODEL_ACQUISITION,"datasetDigest":DATASET_DIGEST,"dataset":DATASET_PROVENANCE,"training":{"method":TRAINING_METHOD,"epochs":EPOCHS,"learningRate":LEARNING_RATE,"loraRank":LORA_RANK},"runtime":{"python":platform.python_version(),"torch":torch.__version__,"gpu":GPU_NAME,"gpuVramGiB":GPU_VRAM_GB},"aiProvenance":AI_PROVENANCE}
(S/"metrics.json").write_text(json.dumps(METRICS,indent=2)); (S/"provenance.json").write_text(json.dumps(PROVENANCE,indent=2)); (S/"MODEL_CARD.md").write_text(f"# PEFT adapter for {model_id}\n\nBase revision: `{revision}`. Optimization metrics are not task-quality evidence.\n")
records=[{"path":p.relative_to(S).as_posix(),"bytes":p.stat().st_size,"sha256":sha256_file(p)} for p in sorted(S.rglob("*")) if p.is_file() and p.name!="artifact-manifest.json"]
(S/"artifact-manifest.json").write_text(json.dumps({"format":"peft_adapter","formatVersion":1,"files":records,"totalBytes":sum(x["bytes"] for x in records)},indent=2))
for r in records:
 if sha256_file(S/r["path"])!=r["sha256"]: raise RuntimeError("artifact-manifest.json verification failed")
# Fresh base + adapter reload, from disk, before publication.
del model,base_model; gc.collect(); torch.cuda.empty_cache()
reload_kw={"trust_remote_code":False,"dtype":dtype,"quantization_config":q,"device_map":{"":0}}
if MODEL_SOURCE=="Pinned Hugging Face": reload_kw.update({"revision":revision,**({"token":HF_TOKEN} if HF_TOKEN else {})})
else: reload_kw["local_files_only"]=True
rb=AutoModelForCausalLM.from_pretrained(MODEL_LOAD_REF,**reload_kw); rt=AutoTokenizer.from_pretrained(S/"tokenizer",local_files_only=True,trust_remote_code=False)
rm=PeftModel.from_pretrained(rb,S,is_trainable=False); tokenizer=rt; smoke=generate(rm,"Kumusta! Sagutin sa isang maikling pangungusap.",32)
if not smoke: raise RuntimeError("Fresh reload failed")
print("✓ Fresh base + adapter reload",smoke); os.replace(S,A)
with zipfile.ZipFile(Z,"w",zipfile.ZIP_STORED) as z:
 for p in A.rglob("*"):
  if p.is_file(): z.write(p,p.relative_to(A).as_posix())
print("Artifact SHA-256",sha256_file(Z))
from google.colab import files
files.download(str(Z))


## 9. What a successful run proves

If the notebook completes, the **exact path you selected** has executor evidence:
1. Pinned base-model acquisition and integrity verification;
2. Structural dataset normalization and split leakage checking;
3. Chat template tokenization with assistant-only loss masking;
4. 4-bit QLoRA fine-tuning and optimization tracking;
5. Before-and-after prompt generation comparison; and
6. Standalone adapter export with verified clean reload.

### Deploying Your Adapter
The exported adapter ZIP can be used in several ways:
- **Inference with PEFT**: Load the adapter on top of the base model using Hugging Face `peft` and `transformers` (demonstrated in our companion inference notebook);
- **Serving Engines**: Deploy using high-throughput serving runtimes like vLLM, SGLang, or Hugging Face TGI, which support dynamic multi-LoRA serving with minimal overhead; or
- **Weight Merging**: Merge the low-rank adapter matrices directly into the base model weights using `model.merge_and_unload()` for zero-overhead, standalone model serving.
